# 유형 06 — 이분탐색 (Binary Search)

> **대상**: 정렬(유형 02)을 익힌 단계.
> **목표**: "정렬된 것에서 절반씩 줄여 찾기"라는 기본기와, 코테의 진짜 핵심인 **파라메트릭 서치(답을 이분탐색)**를 구분해 쓰기.

이분탐색은 **"탐색 범위를 절반씩 버리는"** 전략입니다. O(n) 선형 탐색을 **O(log n)**으로 줄이죠. 핵심 전제는 **단조성(monotonic)** — 한 방향으로 정렬돼 있거나, "어떤 값을 기준으로 조건이 한 번만 바뀌어야" 합니다.


## 0) 언제 이분탐색인가

| 신호 | 형태 |
|------|------|
| "정렬된 배열에서 ~를 찾아라" | 기본 이분탐색 |
| "~를 만족하는 **최소/최대 값**을 구하라" | 파라메트릭 서치 (답을 이분탐색) |
| 입력이 매우 큼 (n ≥ 10만, 값 범위 ~10억) | 선형으론 시간초과 → 이분탐색 의심 |

> 가장 중요한 전환: **"최솟값의 최대 / 최댓값의 최소"** 같은 표현이 나오면 거의 파라메트릭 서치입니다.

## 1) 기본 이분탐색 — 정렬된 배열에서 값 찾기

```python
def binary_search(arr, target):
    lo, hi = 0, len(arr) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if arr[mid] == target:
            return mid           # 찾음
        elif arr[mid] < target:
            lo = mid + 1         # 오른쪽 절반만
        else:
            hi = mid - 1         # 왼쪽 절반만
    return -1                    # 못 찾음

binary_search([1, 3, 5, 7, 9, 11], 7)   # 3 (인덱스)
```

> 패턴: **`lo`/`hi` 두 포인터 → `mid` 확인 → 절반 버리기.** `lo <= hi`와 `mid ± 1`이 무한루프를 막는 핵심.

**표준 라이브러리 `bisect`** (실무/코테에서 자주):

```python
import bisect

arr = [1, 3, 5, 7, 9]
bisect.bisect_left(arr, 5)    # 2  (5가 들어갈 가장 왼쪽 위치)
bisect.bisect_right(arr, 5)   # 3  (5가 들어갈 가장 오른쪽 위치)
# "x 이상인 첫 위치", "x 이하 개수" 등을 O(log n)으로
```

In [3]:
def binary_search(arr, target):
    lo, hi = 0, len(arr) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if arr[mid] == target:
            return mid           # 찾음
        elif arr[mid] < target:
            lo = mid + 1         # 오른쪽 절반만
        else:
            hi = mid - 1         # 왼쪽 절반만
    return -1                    # 못 찾음

binary_search([1, 3, 5, 7, 9, 11], 7)   # 3 (인덱스)

3

In [4]:
import bisect

arr = [1, 3, 5, 7, 9]
bisect.bisect_left(arr, 5)    # 2  (5가 들어갈 가장 왼쪽 위치)
bisect.bisect_right(arr, 5)   # 3  (5가 들어갈 가장 오른쪽 위치)
# "x 이상인 첫 위치", "x 이하 개수" 등을 O(log n)으로

3

## 2) 파라메트릭 서치 — 답 자체를 이분탐색 (코테 핵심)

배열을 뒤지는 게 아니라, **"가능한 답의 범위"를 이분탐색**합니다.

**틀**: "어떤 값 `x`로 했을 때 조건을 만족하는가?"가 **`x`가 커질수록 한 방향으로만 바뀔 때** 쓸 수 있습니다.

```python
# 틀: 만족하는 최소 x 찾기
def parametric(lo, hi, possible):
    answer = hi
    while lo <= hi:
        mid = (lo + hi) // 2
        if possible(mid):       # mid로 가능하면
            answer = mid        # 후보 저장하고
            hi = mid - 1        # 더 작은 값 탐색
        else:
            lo = mid + 1        # mid로 불가능하면 더 큰 값
    return answer
```

> 핵심 사고: **"답이 mid라면 조건이 충족되나?"를 판정하는 함수**를 만들고, 그 판정이 한 번만 뒤집히는 성질(단조성)을 이용해 답의 범위를 절반씩 줄입니다.

In [5]:
def parametric(lo, hi, possible):
    answer = hi
    while lo <= hi:
        mid = (lo + hi) // 2
        if possible(mid):       # mid로 가능하면
            answer = mid        # 후보 저장하고
            hi = mid - 1        # 더 작은 값 탐색
        else:
            lo = mid + 1        # mid로 불가능하면 더 큰 값
    return answer

## 대표 문제 풀이 — 입국심사 (프로그래머스 43238)

> 심사관별 심사 시간 `times`가 주어진다. `n`명을 모두 심사하는 데 걸리는 **최소 시간**을 구하라. (심사관들은 동시에 일함)

**① 신호**: "n명을 심사하는 **최소 시간**" — 답(시간)을 이분탐색
**② 파라메트릭 서치**: "시간 `t` 안에 `n`명 이상 처리 가능한가?" → `t`가 클수록 처리 인원이 늘어남(단조성)

```python
def solution(n, times):
    # 한 명도 못 끝낸 0부터, 가장 느린 심사관이 n명 다 보는 시간까지가 답의 범위
    lo, hi = 1, max(times) * n
    answer = hi
    while lo <= hi:
        mid = (lo + hi) // 2
        # mid 시간 동안 처리 가능한 총 인원
        people = sum(mid // t for t in times)
        if people >= n:         # n명 이상 가능하면
            answer = mid        # 이 시간을 후보로 저장
            hi = mid - 1        # 더 줄여본다
        else:
            lo = mid + 1        # 부족하면 시간을 늘린다
    return answer
```

**검증:**

```python
print(solution(6, [7, 10]))     # 28

assert solution(6, [7, 10]) == 28
assert solution(1, [1]) == 1
assert solution(3, [3]) == 9     # 심사관 1명이 3명 → 3*3
print("통과 ✅")
```

> 가르칠 포인트:
> 1. **이분탐색 대상이 "배열"이 아니라 "시간(답)"** — 1부터 `max(times)*n`까지가 답이 존재할 수 있는 범위.
> 2. **판정 함수 `sum(mid // t for t in times)`** — `mid` 시간 동안 각 심사관이 처리하는 인원의 합. 이게 `n` 이상이면 "그 시간이면 충분"하니 더 줄여본다.
> 3. **단조성** — 시간이 늘면 처리 인원은 절대 줄지 않음 → 이분탐색 성립.

In [6]:
def solution(n, times):
    # 한 명도 못 끝낸 0부터, 가장 느린 심사관이 n명 다 보는 시간까지가 답의 범위
    lo, hi = 1, max(times) * n
    answer = hi
    while lo <= hi:
        mid = (lo + hi) // 2
        # mid 시간 동안 처리 가능한 총 인원
        people = sum(mid // t for t in times)
        if people >= n:         # n명 이상 가능하면
            answer = mid        # 이 시간을 후보로 저장
            hi = mid - 1        # 더 줄여본다
        else:
            lo = mid + 1        # 부족하면 시간을 늘린다
    return answer

In [7]:
print(solution(6, [7, 10]))     # 28

assert solution(6, [7, 10]) == 28
assert solution(1, [1]) == 1
assert solution(3, [3]) == 9     # 심사관 1명이 3명 → 3*3
print("통과 ✅")

28
통과 ✅


## 직접 풀어보기 (연습)

힌트를 보고 `TODO`를 채운 뒤 검증 셀을 실행하세요. **"값을 찾기"인지 "답을 이분탐색"인지**부터 구분합니다.

### 연습 1 — 정렬된 배열에서 위치 찾기

> 오름차순 정렬된 `arr`에서 `target`의 인덱스를 반환하라. 없으면 -1.
> **힌트**: 기본 이분탐색. `lo<=hi` 동안 `mid`를 확인하고 절반씩 버리기.

```python
def solution(arr, target):
    # TODO: 직접 작성
    pass

# 검증
assert solution([1, 3, 5, 7, 9], 7) == 3
assert solution([1, 3, 5, 7, 9], 4) == -1
assert solution([10], 10) == 0
print("통과 ✅")
```

### 연습 2 — 떡 자르기 (파라메트릭 서치)

> 떡 길이 리스트 `heights`. 절단기 높이 `H`로 자르면, `H`보다 큰 부분만 잘려 나간다. 잘린 떡 길이의 합이 `target` 이상이 되게 하는 **절단기의 최대 높이 `H`**를 구하라.
> **힌트**: 높이 `H`를 이분탐색. `H`가 낮을수록 많이 잘림(단조성). 합 `≥ target`인 **가장 큰 `H`**를 찾기.

```python
def solution(heights, target):
    # TODO: H를 이분탐색 (0 ~ max(heights))
    pass

# 검증
assert solution([19, 15, 10, 17], 6) == 15   # H=15 → (19-15)+(17-15)=6
assert solution([4, 42, 40, 26, 46], 20) == 36
print("통과 ✅")
```

### 연습 3 — 두 배열의 공통 원소 개수 (bisect 활용, 도전)

> 정렬된 배열 `a`와 임의 배열 `b`가 주어진다. `b`의 각 원소가 `a`에 있는지 이분탐색으로 확인해, `a`에도 있는 `b` 원소의 개수를 구하라.
> **힌트**: `a`를 정렬해 두고, `b`의 각 값을 `bisect_left`로 위치를 찾아 실제 존재하는지 확인. (선형 `in`은 O(n²)라 큰 입력에서 느림)

```python
import bisect

def solution(a, b):
    # TODO: bisect로 직접 작성
    pass

# 검증
assert solution([1, 3, 5, 7, 9], [3, 4, 5, 10]) == 2   # 3, 5
assert solution([2, 4, 6], [1, 3, 5]) == 0
print("통과 ✅")
```